In [1]:
import pandas as pd
from tabulate import tabulate

class RetirementPlanner:
    def __init__(self, inputs):
        self.inputs = inputs
        # Unpack commonly used variables for easier access
        self.lump_sum = inputs['lump_sum']
        self.horizon_months = int(inputs['horizon_years'] * 12)

    def calculate_swr_health(self):
        """Calculates Safe Withdrawal Rate health check."""
        initial_gap = self.inputs['target_payout'] - (self.inputs['other_income'] + self.inputs['current_dividends'])
        if initial_gap <= 0: return 0.0, "SECURE"

        gross_annual_withdrawal = (initial_gap * 12) / (1 - self.inputs['tax_rate'])

        if self.lump_sum <= 0: return 0.0, "CRITICAL"

        actual_withdrawal_rate = gross_annual_withdrawal / self.lump_sum

        if actual_withdrawal_rate <= 0.035: status = "EXCELLENT (< 3.5%)"
        elif actual_withdrawal_rate <= 0.045: status = "GOOD (Industry Std)"
        elif actual_withdrawal_rate <= 0.06: status = "RISKY (High Depletion Risk)"
        else: status = "DANGER (> 6.0%)"

        return actual_withdrawal_rate, status

    def calculate_viability(self, crash_rate=0.0, crash_years=0):
        """
        Runs the simulation month-by-month.
        """
        # Initialize simulation variables
        balance = self.lump_sum
        current_div = self.inputs['current_dividends']
        current_expense = self.inputs['target_payout']

        data = []
        is_solvent = True
        insolvency_msg = "Solvent"

        # Monthly Loop
        for m in range(1, self.horizon_months + 1):
            year = (m - 1) // 12 + 1

            # --- 1. DETERMINE MONTHLY RETURN ---
            # If in crash period, use crash rate. Otherwise, use standard return.
            if year <= crash_years:
                monthly_rate = crash_rate / 12
            else:
                monthly_rate = self.inputs['expected_return'] / 12

            # --- 2. ANNUAL INFLATION ADJUSTMENTS ---
            if m > 1 and (m - 1) % 12 == 0:
                current_div *= (1 + self.inputs['gdp_growth'])
                current_expense *= (1 + self.inputs['inflation_rate'])

            # --- 3. CORE FINANCIAL LOGIC ---
            # A. Earn Interest
            balance += (balance * monthly_rate)

            # B. Calculate Gap
            total_income = self.inputs['other_income'] + current_div
            net_shortfall = max(0, current_expense - total_income)

            # C. Tax Gross Up (Withdrawal = Net / (1 - TaxRate))
            gross_withdrawal = net_shortfall / (1 - self.inputs['tax_rate'])

            # D. Withdraw
            balance -= gross_withdrawal

            # E. Solvency Check
            if balance <= 0 and is_solvent:
                is_solvent = False
                insolvency_msg = f"DEPLETED (Year {year})"
                balance = 0

            # F. Snapshot (End of Year)
            if m % 12 == 0 or m == self.horizon_months:
                data.append({
                    "Year": year,
                    "Expenses (Net)": current_expense,
                    "Withdrawal (Gross)": gross_withdrawal,
                    "Balance": balance
                })

        return pd.DataFrame(data), is_solvent, insolvency_msg

    def solve_required_capital(self, crash_rate=0, crash_years=0):
        """Binary search to find exact capital needed to avoid depletion."""
        original_balance = self.lump_sum
        low = self.lump_sum
        high = self.lump_sum * 10

        for _ in range(30): # 30 iterations is enough precision
            mid = (low + high) / 2
            self.lump_sum = mid
            _, solvent, _ = self.calculate_viability(crash_rate, crash_years)
            if solvent: high = mid
            else: low = mid

        required = high
        self.lump_sum = original_balance # Restore original
        return required

# --- Helper Functions ---

def get_valid_input(prompt, type_func=float, is_percentage=False, allow_negative=False):
    while True:
        try:
            val = input(prompt)
            if not val: continue
            num = type_func(val)
            if not allow_negative and num < 0:
                print(">> Must be positive.")
                continue
            if is_percentage and abs(num) > 1: return num / 100
            return num
        except ValueError:
            print(">> Invalid number.")

def build_portfolio():
    print("\n" + "="*40)
    print(" PORTFOLIO ARCHITECT")
    print("="*40)
    print("1. Aggressive Preset (60% Eq, 20% Met, 10% Dbt, 10% Cash)")
    print("2. Conservative Preset (20% Eq, 10% Met, 40% Dbt, 30% Cash)")
    print("3. Build Custom Allocation")

    choice = input("\nSelect Portfolio Strategy (1-3): ")

    alloc = {}

    if choice == '1':
        alloc = {'Equity': 0.60, 'Metals': 0.20, 'Debt': 0.10, 'Cash/Fix': 0.10}
        est_return = 0.08 # 8% historical proxy
    elif choice == '2':
        alloc = {'Equity': 0.20, 'Metals': 0.10, 'Debt': 0.40, 'Cash/Fix': 0.30}
        est_return = 0.05 # 5% historical proxy
    else:
        print("\n>> Define your Asset Mix (Must sum to 100%):")
        while True:
            e = get_valid_input("  % Equity (Stocks): ", is_percentage=True)
            m = get_valid_input("  % Metals (Gold/Silver): ", is_percentage=True)
            d = get_valid_input("  % Debt (Bonds): ", is_percentage=True)
            c = get_valid_input("  % Cash/Fixed Income: ", is_percentage=True)

            total = e + m + d + c
            if 0.99 <= total <= 1.01:
                alloc = {'Equity': e, 'Metals': m, 'Debt': d, 'Cash/Fix': c}
                # Weighted Average Return Estimation
                # Stocks~9%, Metals~4%, Bonds~4%, Cash~2%
                est_return = (e*0.09) + (m*0.04) + (d*0.04) + (c*0.02)
                break
            else:
                print(f">> Total is {total*100:.1f}%. Please make it 100%.")

    print(f"\n[ System Analysis ]")
    print(f"Based on your allocation, the estimated historical return is: {est_return*100:.2f}%")
    override = input("Press ENTER to accept, or type a new return rate (e.g. 7): ")
    if override:
        est_return = float(override)
        if est_return > 1: est_return /= 100

    return alloc, est_return

def suggest_tickers(allocation):
    print("\n--- Asset Class Ticker Suggestions ---")
    data = []
    if allocation['Equity'] > 0:
        data.append(["Equity", f"{allocation['Equity']*100:.0f}%", "VTI (Total US), VEU (Global), VIG (Div Growth)"])
    if allocation['Metals'] > 0:
        data.append(["Metals", f"{allocation['Metals']*100:.0f}%", "GLD (Gold), SLV (Silver), IAU"])
    if allocation['Debt'] > 0:
        data.append(["Debt", f"{allocation['Debt']*100:.0f}%", "BND (Total Bond), TLT (Long Term Treasuries)"])
    if allocation['Cash/Fix'] > 0:
        data.append(["Cash/Fix", f"{allocation['Cash/Fix']*100:.0f}%", "SGOV (T-Bills), HYSA, CD Ladders"])

    print(tabulate(data, headers=["Class", "Alloc", "Ticker Ideas"], tablefmt="simple"))

# --- Main Execution ---
if __name__ == "__main__":
    print("\n" + "*"*60)
    print(" RETIREMENT ENGINE v8.0 (Portfolio Architect)")
    print("*"*60)

    # 1. Financial Inputs
    i = {}
    print("\n[ STEP 1: FINANCIAL DATA ]")
    i['lump_sum'] = get_valid_input("Total Investment Capital ($): ")
    i['other_income'] = get_valid_input("Monthly Fixed Income ($): ")
    i['current_dividends'] = get_valid_input("Monthly Dividend Income ($): ")
    i['target_payout'] = get_valid_input("Target Monthly Spend (Net) ($): ")
    i['tax_rate'] = get_valid_input("Tax Rate (e.g. 15): ", is_percentage=True)
    i['horizon_years'] = get_valid_input("Planning Horizon (Years): ", int)
    i['inflation_rate'] = get_valid_input("Inflation Rate (e.g. 2.5): ", is_percentage=True)
    i['gdp_growth'] = get_valid_input("Dividend Growth Rate (e.g. 3.0): ", is_percentage=True)

    # 2. Portfolio Construction
    allocation, expected_ret = build_portfolio()
    i['expected_return'] = expected_ret

    suggest_tickers(allocation)

    # 3. Initialize Planner
    planner = RetirementPlanner(i)

    # 4. Standard Simulation
    print("\n" + "="*60)
    print(f" BASELINE SIMULATION ({expected_ret*100:.2f}% Return)")
    print("="*60)
    df, solvent, msg = planner.calculate_viability()

    swr, swr_status = planner.calculate_swr_health()
    print(f"Initial Withdrawal Rate: {swr*100:.2f}% -> {swr_status}")
    print(f"Outcome: {msg}")

    if not solvent:
        req = planner.solve_required_capital()
        print(f">> FIX: You need ${req:,.0f} (Total) to make this plan work.")

    if df is not None:
        print("\n--- Year-by-Year (First 15 Years) ---")
        pd.options.display.float_format = '${:,.0f}'.format
        print(tabulate(df.head(15), headers='keys', tablefmt='simple', showindex=False))

    # 5. Stress Test Loop
    while True:
        print("\n" + "-"*60)
        choice = input(">> Run a Stress Test Scenario? (y/n): ").lower()
        if choice != 'y':
            print("Exiting. Good luck!")
            break

        print("\n[ STRESS TEST CONFIG ]")
        crash_pct = get_valid_input("  Crash Annual Return (e.g. -20 for -20%): ", is_percentage=True, allow_negative=True)
        crash_yrs = get_valid_input("  Crash Duration (Years): ", int)

        df_stress, solv_stress, msg_stress = planner.calculate_viability(crash_pct, crash_yrs)

        print(f"\n[ RESULT: {crash_pct*100:.0f}% Crash for {crash_yrs} Years ]")
        print(f"Outcome: {msg_stress}")

        if not solv_stress:
             req_stress = planner.solve_required_capital(crash_pct, crash_yrs)
             diff = req_stress - i['lump_sum']
             print(f"Solution: To survive this crash, you need ${req_stress:,.0f} (+${diff:,.0f})")

             # Show the failure point
             print("\n--- Failure Trajectory ---")
             print(tabulate(df_stress[['Year', 'Withdrawal (Gross)', 'Balance']].head(15), headers='keys', tablefmt='simple', showindex=False))


************************************************************
 RETIREMENT ENGINE v8.0 (Portfolio Architect)
************************************************************

[ STEP 1: FINANCIAL DATA ]
Total Investment Capital ($): 10000000
Monthly Fixed Income ($): 25000
Monthly Dividend Income ($): 5000
Target Monthly Spend (Net) ($): 100000
Tax Rate (e.g. 15): 12.5
Planning Horizon (Years): 35
Inflation Rate (e.g. 2.5): 1.75
Dividend Growth Rate (e.g. 3.0): 2.5

 PORTFOLIO ARCHITECT
1. Aggressive Preset (60% Eq, 20% Met, 10% Dbt, 10% Cash)
2. Conservative Preset (20% Eq, 10% Met, 40% Dbt, 30% Cash)
3. Build Custom Allocation

Select Portfolio Strategy (1-3): 1

[ System Analysis ]
Based on your allocation, the estimated historical return is: 8.00%
Press ENTER to accept, or type a new return rate (e.g. 7): 

--- Asset Class Ticker Suggestions ---
Class     Alloc    Ticker Ideas
--------  -------  ----------------------------------------------
Equity    60%      VTI (Total US), VEU (Global